In [67]:
import pandas as pd
import numpy as np

In [68]:
df = pd.read_parquet("data/clean/us_equities.parquet")
print(df.head())
print(df.shape)

   ^GSPC  GE  IBM  DIS  BA  CAT  AA  HPQ  DD  KO  ...  NSM  CLP  CTX  CTR  \
0  16.66 NaN  NaN  NaN NaN  NaN NaN  NaN NaN NaN  ...  NaN  NaN  NaN  NaN   
1  16.85 NaN  NaN  NaN NaN  NaN NaN  NaN NaN NaN  ...  NaN  NaN  NaN  NaN   
2  16.93 NaN  NaN  NaN NaN  NaN NaN  NaN NaN NaN  ...  NaN  NaN  NaN  NaN   
3  16.98 NaN  NaN  NaN NaN  NaN NaN  NaN NaN NaN  ...  NaN  NaN  NaN  NaN   
4  17.08 NaN  NaN  NaN NaN  NaN NaN  NaN NaN NaN  ...  NaN  NaN  NaN  NaN   

   DYN  AIB  KIM  SFN  TCO   S  
0  NaN  NaN  NaN  NaN  NaN NaN  
1  NaN  NaN  NaN  NaN  NaN NaN  
2  NaN  NaN  NaN  NaN  NaN NaN  
3  NaN  NaN  NaN  NaN  NaN NaN  
4  NaN  NaN  NaN  NaN  NaN NaN  

[5 rows x 1012 columns]
(16155, 1012)


In [69]:
# drop rows with more than 50 % missing values
df = df.dropna(thresh=df.shape[1] // 2)
df = df.reset_index(drop=True)
df = df.interpolate(method='linear', axis=0).ffill().bfill()
df.shape

(6088, 1012)

In [70]:
N = 100
t_0 = 1050
t_1 = 1651

df_sampled = df[t_0:t_1].iloc[:, :N]
print(df_sampled.isna().sum().sum())

0


In [71]:
df_sampled.head()

,^GSPC,GE,IBM,DIS,BA,CAT,AA,HPQ,DD,KO,...,LPX,VLO,WMB,TXI,CI,NVO,OMX,NSC,ALK,CLX
1050,466.91,5.12,10.33,12.45,16.49,8.90,6.57,6.65,13.82,6.82,...,26.01,1.13,3.32,11.82,5.58,0.75,15.11,11.68,8.18,7.94
1051,465.88,5.08,10.82,12.45,16.23,8.93,6.51,6.65,13.79,6.72,...,25.63,1.11,3.29,11.16,5.49,0.76,15.18,11.85,8.11,8.02
1052,467.06,5.10,10.95,12.82,16.14,8.94,6.52,6.73,13.98,6.73,...,25.63,1.08,3.32,11.08,5.54,0.76,14.97,11.74,8.36,8.07
1053,463.90,5.06,11.07,12.59,16.19,8.97,6.36,6.80,14.34,6.71,...,24.27,1.10,3.48,11.20,5.44,0.78,14.69,11.70,8.24,8.06
1054,466.44,5.09,11.04,12.62,16.45,9.09,6.44,6.83,14.54,6.73,...,24.42,1.13,3.37,11.20,5.49,0.77,14.69,11.74,8.36,8.04


In [72]:
e = np.ones(N)
cov_mat = df_sampled.cov()
inv_cov_mat = np.linalg.inv(cov_mat.values)
w = inv_cov_mat @ e / (e.T @ inv_cov_mat @ e)
calibration_risk = w.T @ cov_mat.values @ w

In [73]:
Delta = 45 
df_realized = df[t_1:t_1+Delta].iloc[:, :N]
realized_cov_mat = df_realized.cov()
realized_risk = w.T @ realized_cov_mat.values @ w

In [74]:
eigval, eigvec = np.linalg.eigh(cov_mat.values)
cov_mat_clipped = np.zeros_like(cov_mat.values)
threshold = eigval[N//4]
for i in range(N):
    cov_mat_clipped += max(threshold, eigval[i]) * np.outer(eigvec[:, i], eigvec[:, i])
inv_cov_mat_clipped = np.linalg.inv(cov_mat_clipped)
w_clipped = inv_cov_mat_clipped @ e / (e.T @ inv_cov_mat_clipped @ e)
calibration_risk_clipped = w_clipped.T @ cov_mat.values @ w_clipped

In [75]:
print(f"Realized risk: {realized_risk}")
print(f"Calibration risk: {calibration_risk}")
print(f"Calibration risk (clipped): {calibration_risk_clipped}")

Realized risk: 0.0006191485623016075
Calibration risk: 8.09118551618669e-05
Calibration risk (clipped): 0.00019861460334643177


In [76]:
N = 250
T = 90
t_0 = 4000
t_1 = 5800

df_sampled = df[t_0:t_1].iloc[:, :N]
df_sampled = df_sampled.reset_index(drop=True)
df_sampled.shape

(1800, 250)

In [77]:
df_sampled.head()

,^GSPC,GE,IBM,DIS,BA,CAT,AA,HPQ,DD,KO,...,MHP,GSK,GGG,TRV,STT,PGR,TSN,NYT,BDN,CQB
0,1257.48,26.47,77.34,22.12,57.23,46.71,24.01,26.74,31.82,16.94,...,44.82,33.69,30.83,37.58,53.33,23.47,15.42,24.49,17.02,21.25
1,1249.48,26.32,77.17,21.99,56.45,47.27,23.61,26.48,31.33,16.80,...,44.74,33.44,30.81,37.44,52.18,23.31,15.37,24.22,17.18,20.42
2,1264.67,26.34,77.44,22.01,57.68,48.26,24.36,26.38,31.74,16.86,...,45.13,34.05,31.48,37.46,52.57,23.48,15.67,24.12,17.32,21.01
3,1265.08,26.16,76.95,21.94,57.49,48.11,24.21,26.09,31.63,16.85,...,45.10,34.43,31.73,37.42,52.48,23.44,15.40,23.84,17.19,20.73
4,1262.09,26.36,76.76,22.06,57.29,47.72,24.09,26.59,31.58,16.78,...,44.77,34.39,31.70,36.51,52.87,23.40,15.09,23.64,16.87,20.42


In [78]:
for i in range(1000):
    df_sampled_i = df_sampled[i:T+i]
    cov_mat = df_sampled_i.cov()